## Imports

In [ ]:
import torch
from torch import nn
import pandas as pd
import numpy as np

## Load data

In [ ]:
TELEMETRY_CSV = "telemetry.csv"
GROUND_TRUTH_CSV = "ground_truth.csv"

telemetry_df = pd.read_csv(TELEMETRY_CSV)
gt_df = pd.read_csv(GROUND_TRUTH_CSV)

print(telemetry_df.shape, gt_df.shape)
print(telemetry_df.columns.tolist())
print(gt_df.columns.tolist())

## Merge

In [ ]:
merged_df = pd.merge(
    telemetry_df,
    gt_df,
    on=["run_id", "t"],
    how="inner",
    validate="one_to_one",
)

assert len(merged_df) == len(telemetry_df), "Row count changed after merge — check join key alignment"
print(merged_df.shape)

## Column groups

In [ ]:
# edit after locking df schema

SENSOR_COLUMNS = [
    "rpm", "torque", "power", "engine_load",
    "cht_c1", "cht_c2", "cht_c3", "cht_c4",
    "egt_c1", "egt_c2", "egt_c3", "egt_c4",
    "oil_pressure", "oil_temperature",
    "fuel_flow", "fuel_quantity_used", "fuel_remaining",
    "rail_pressure", "injection_timing", "injection_duration",
    "boost_pressure", "map", "intake_temperature", "air_mass_flow", "turbo_speed",
    "coolant_temperature",
    "vibration_rms_x", "vibration_rms_y", "vibration_rms_z", "vibration_order_1x",
    "battery_voltage", "battery_current", "alternator_power",
    "altitude", "ambient_pressure", "ambient_temperature", "air_density",
    "throttle", "load_demand",
]

CATEGORICAL_COLUMNS = ["engine_state"]

HEALTH_1_TO_0_COLUMNS = [
    "injector_health_c1", "injector_health_c2", "injector_health_c3", "injector_health_c4",
    "cooling_health", "oil_pump_health", "bearing_health",
    "fuel_delivery_health", "alternator_health",
]
HEALTH_0_TO_1_COLUMNS = [
    "turbo_efficiency_deg", "injection_timing_deg", "combustion_stability",
    "misfire_rate_c1", "misfire_rate_c2", "misfire_rate_c3", "misfire_rate_c4",
]

SENSOR_FAULT_COLUMNS = [c for c in gt_df.columns if c.startswith("sensor_fault_active_")]
print("Detected sensor-fault columns:", SENSOR_FAULT_COLUMNS)

## Health sign flip

In [ ]:
for col in HEALTH_0_TO_1_COLUMNS:
    merged_df[col] = 1.0 - merged_df[col]

HEALTH_ALL_COLUMNS = HEALTH_1_TO_0_COLUMNS + HEALTH_0_TO_1_COLUMNS
print(f"{len(HEALTH_ALL_COLUMNS)} health columns, all now 1.0=healthy -> 0.0=failed")

merged_df[HEALTH_ALL_COLUMNS].describe().T[["min", "max"]]

## Failure thresholds

In [ ]:
# Map each (now-flipped) health column to its failure threshold, converted into the
# same 1.0=healthy->0.0=failed direction as Cell 4.
#
# Original criteria from failure-mode-matrix.csv, converted:
#   injector_health_c{n} <= 0.6      -> already this direction, threshold = 0.6
#   oil_pump_health <= 0.5           -> threshold = 0.5
#   cooling_health <= 0.5            -> threshold = 0.5
#   turbo_efficiency_deg >= 0.7      -> flipped: 1-0.7 = 0.3
#   bearing_health <= 0.5            -> threshold = 0.5
#   misfire_rate >= 0.1              -> flipped: 1-0.1 = 0.9
#   combustion_stability >= 0.5      -> flipped: 1-0.5 = 0.5
#   fuel_delivery_health <= 0.6      -> threshold = 0.6
#   alternator_health <= 0.5         -> threshold = 0.5
#   injection_timing_deg >= 0.6      -> flipped: 1-0.6 = 0.4

FAILURE_THRESHOLDS = {
    "injector_health_c1": 0.6, "injector_health_c2": 0.6,
    "injector_health_c3": 0.6, "injector_health_c4": 0.6,
    "oil_pump_health": 0.5,
    "cooling_health": 0.5,
    "bearing_health": 0.5,
    "fuel_delivery_health": 0.6,
    "alternator_health": 0.5,
    "turbo_efficiency_deg": 0.3,        # flipped from >=0.7
    "injection_timing_deg": 0.4,        # flipped from >=0.6
    "combustion_stability": 0.5,        # flipped from >=0.5
    "misfire_rate_c1": 0.9,             # flipped from >=0.1
    "misfire_rate_c2": 0.9,
    "misfire_rate_c3": 0.9,
    "misfire_rate_c4": 0.9,
}

missing = set(HEALTH_ALL_COLUMNS) - set(FAILURE_THRESHOLDS)
assert not missing, f"Missing thresholds for: {missing}"

## RUL deviation

In [ ]:
def compute_rul_for_run(run_df: pd.DataFrame) -> pd.Series:
    """
    For each timestep, RUL = time until the FIRST future crossing of any health
    parameter below its failure threshold. If a parameter never crosses within
    the run, it does not contribute (treated as right-censored / no failure event
    for that parameter in this run).
    """
    run_df = run_df.sort_values("t").reset_index(drop=True)
    t = run_df["t"].values
    n = len(run_df)
    rul = np.full(n, np.nan)

    # For each health column, find the first index where it crosses its threshold
    first_failure_times = []
    for col, thresh in FAILURE_THRESHOLDS.items():
        below = run_df[col].values <= thresh
        idx = np.argmax(below) if below.any() else -1
        if idx > 0 or (idx == 0 and below[0]):
            first_failure_times.append(t[idx])

    if len(first_failure_times) == 0:
        # No failure reached in this run — RUL = time remaining to end of run
        # (censored; flag separately if you want to exclude these from RUL loss)
        end_time = t[-1]
        rul = end_time - t
    else:
        nearest_failure_t = min(first_failure_times)
        rul = np.clip(nearest_failure_t - t, a_min=0, a_max=None)

    return pd.Series(rul, index=run_df.index)


rul_parts = []
for run_id, run_df in merged_df.groupby("run_id", sort=False):
    rul_series = compute_rul_for_run(run_df)
    rul_series.index = run_df.index
    rul_parts.append(rul_series)

merged_df["rul"] = pd.concat(rul_parts).sort_index()
merged_df[["run_id", "t", "rul"]].head(10)

## Sensor fault class mapping

In [ ]:
SENSOR_FAULT_CLASSES = ["NONE", "BIAS", "DRIFT", "NOISE", "STUCK", "DROPOUT"]
sensor_fault_class_map = {c: i for i, c in enumerate(SENSOR_FAULT_CLASSES)}

for col in SENSOR_FAULT_COLUMNS:
    merged_df[col] = merged_df[col].map(sensor_fault_class_map)

merged_df[SENSOR_FAULT_COLUMNS].head()

## Train + val split + scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

# Simple run-level split; replace with your actual train/val/test run_id assignment
unique_runs = merged_df["run_id"].unique()
np.random.seed(42)
np.random.shuffle(unique_runs)
n_train = int(0.8 * len(unique_runs))
train_runs, val_runs = unique_runs[:n_train], unique_runs[n_train:]

train_mask = merged_df["run_id"].isin(train_runs)

scaler = StandardScaler()
scaler.fit(merged_df.loc[train_mask, SENSOR_COLUMNS])  # fit stats on training runs only

merged_df[SENSOR_COLUMNS] = scaler.transform(merged_df[SENSOR_COLUMNS])

## Encoder

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import joblib

ENGINE_STATE_CATEGORIES = [
    "OFF", "STARTING", "IDLE", "TAKEOFF", "CLIMB", "CRUISE",
    "HIGH_ALTITUDE_CRUISE", "LOITER", "THROTTLE_TRANSIENT",
    "DESCENT", "SHUTDOWN", "FAULT",
]

engine_state_encoder = OneHotEncoder(
    categories=[ENGINE_STATE_CATEGORIES],
    handle_unknown="ignore",
    sparse_output=False,
)

engine_state_encoder.fit(merged_df.loc[train_mask, ["engine_state"]])

engine_state_encoded = engine_state_encoder.transform(merged_df[["engine_state"]])
engine_state_col_names = engine_state_encoder.get_feature_names_out(["engine_state"]).tolist()

engine_state_df = pd.DataFrame(engine_state_encoded, columns=engine_state_col_names, index=merged_df.index)
merged_df = pd.concat([merged_df, engine_state_df], axis=1)

joblib.dump(engine_state_encoder, "engine_state_encoder.joblib")

print("engine_state columns:", engine_state_col_names)

## FEATURE_COLUMNS

In [ ]:
FEATURE_COLUMNS = SENSOR_COLUMNS + engine_state_col_names
TARGET_HEALTH_COLUMNS = HEALTH_ALL_COLUMNS      # regression, Head A
TARGET_RUL_COLUMN = "rul"                        # regression, Head B
TARGET_SENSOR_FAULT_COLUMNS = SENSOR_FAULT_COLUMNS  # classification, Head C

print(f"{len(FEATURE_COLUMNS)} input features")
print(f"{len(TARGET_HEALTH_COLUMNS)} health targets")
print(f"{len(TARGET_SENSOR_FAULT_COLUMNS)} sensor-fault targets")

## Build windows

In [ ]:
def build_windows(df, feature_cols, health_cols, rul_col, sensor_fault_cols,
                   seq_len=60, stride=1):
    X_list, y_health_list, y_rul_list, y_sensor_fault_list = [], [], [], []

    for run_id, run_df in df.groupby("run_id", sort=False):
        run_df = run_df.sort_values("t").reset_index(drop=True)
        n = len(run_df)
        if n < seq_len:
            continue

        feats = run_df[feature_cols].values
        health = run_df[health_cols].values
        rul = run_df[rul_col].values
        sensor_fault = run_df[sensor_fault_cols].values

        for start in range(0, n - seq_len + 1, stride):
            end = start + seq_len
            X_list.append(feats[start:end])
            # use the LAST timestep in the window as the label point
            y_health_list.append(health[end - 1])
            y_rul_list.append(rul[end - 1])
            y_sensor_fault_list.append(sensor_fault[end - 1])

    return (
        np.array(X_list, dtype=np.float32),
        np.array(y_health_list, dtype=np.float32),
        np.array(y_rul_list, dtype=np.float32),
        np.array(y_sensor_fault_list, dtype=np.int64),
    )


train_df = merged_df[train_mask]
val_df = merged_df[~train_mask]

X_train, y_health_train, y_rul_train, y_sensor_fault_train = build_windows(
    train_df, FEATURE_COLUMNS, TARGET_HEALTH_COLUMNS, TARGET_RUL_COLUMN, TARGET_SENSOR_FAULT_COLUMNS
)
X_val, y_health_val, y_rul_val, y_sensor_fault_val = build_windows(
    val_df, FEATURE_COLUMNS, TARGET_HEALTH_COLUMNS, TARGET_RUL_COLUMN, TARGET_SENSOR_FAULT_COLUMNS
)

print("Train:", X_train.shape, y_health_train.shape, y_rul_train.shape, y_sensor_fault_train.shape)
print("Val:  ", X_val.shape, y_health_val.shape, y_rul_val.shape, y_sensor_fault_val.shape)

## Sanity checks

In [ ]:
# Health values should be in [0,1] after the sign-flip in Cell 4
assert np.nanmin(y_health_train) >= -1e-3 and np.nanmax(y_health_train) <= 1.0 + 1e-3

# RUL should be non-negative
assert np.nanmin(y_rul_train) >= 0

# Sensor-fault class ids should be within range
assert y_sensor_fault_train.min() >= 0 and y_sensor_fault_train.max() < len(SENSOR_FAULT_CLASSES)

print("All sanity checks passed.")

## Set device

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'